In [0]:
from dotenv import load_dotenv
import os

load_dotenv(".env")

client_id = os.getenv("ADLS_CLIENT_ID")
tenant_id = os.getenv("ADLS_TENANT_ID")
client_secret = os.getenv("ADLS_CLIENT_SECRET")
storage_account = os.getenv("ADLS_STORAGE_ACCOUNT_NAME")

print("client_id carregado:", client_id is not None)
print("tenant_id carregado:", tenant_id is not None)
print("client_secret carregado:", client_secret is not None)
print("storage_account carregado:", storage_account is not None)

In [0]:
# Configurações OAuth do Service Principal para acesso ao ADLS

adls_options = {
    f"fs.azure.account.auth.type.{storage_account}.dfs.core.windows.net": "OAuth",

    f"fs.azure.account.oauth.provider.type.{storage_account}.dfs.core.windows.net":
        "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",

    f"fs.azure.account.oauth2.client.id.{storage_account}.dfs.core.windows.net":
        client_id,

    f"fs.azure.account.oauth2.client.secret.{storage_account}.dfs.core.windows.net":
        client_secret,

    f"fs.azure.account.oauth2.client.endpoint.{storage_account}.dfs.core.windows.net":
        f"https://login.microsoftonline.com/{tenant_id}/oauth2/token"
}

print("✅ Configuração do ADLS preparada com sucesso.")

In [0]:
# ============================================
# CONFIGURAÇÃO DO POLLING - ECOMMERCE_CLIENTES
# ============================================

TABELA = "ecommerce_clientes"
CAMADA = "bronze"

INTERVALO_POLLING = 60  # segundos

print("Tabela:", TABELA)
print("Camada:", CAMADA)
print("Intervalo do polling:", INTERVALO_POLLING, "segundos")

In [0]:
from pyspark.sql import functions as F

def adicionar_particoes_atualizacao(df):
    return (
        df
        .withColumn(
            "ano",
            F.year(F.col("dt_ultima_atualizacao"))
        )
        .withColumn(
            "mes",
            F.month(F.col("dt_ultima_atualizacao"))
        )
        .withColumn(
            "dia",
            F.dayofmonth(F.col("dt_ultima_atualizacao"))
        )
    )

print("✅ Helper de particionamento criada")

In [0]:
container_squad = "squad1"

In [0]:
def montar_caminho_delta(container_squad, camada, tabela):
    """
    Monta o caminho ADLS para as camadas Bronze, Silver ou Gold.
    """
    
    return (
        f"abfss://{container_squad}@{storage_account}.dfs.core.windows.net/"
        f"{camada}/{tabela}"
    )

In [0]:
bronze_path = montar_caminho_delta(
    container_squad,
    "bronze",
    "ecommerce_clientes"
)

print("Caminho Bronze:")
print(bronze_path)

In [0]:
silver_path = montar_caminho_delta(
    container_squad,
    "silver",
    "ecommerce_clientes"
)

print("Caminho Silver:")
print(silver_path)

In [0]:
from azure.identity import ClientSecretCredential
from azure.storage.filedatalake import DataLakeServiceClient

credential = ClientSecretCredential(
    tenant_id=tenant_id,
    client_id=client_id,
    client_secret=client_secret
)

service_client = DataLakeServiceClient(
    account_url=f"https://{storage_account}.dfs.core.windows.net",
    credential=credential
)

print("Containers disponíveis:")

for filesystem in service_client.list_file_systems():
    print(filesystem["name"])

In [0]:
# Procurar ecommerce_clientes dentro do container RAW

file_system_client = service_client.get_file_system_client(
    file_system="raw"
)

arquivos_clientes = []

for arquivo in file_system_client.get_paths():
    caminho = arquivo.name

    if "ecommerce_clientes" in caminho.lower():
        arquivos_clientes.append(caminho)

print("===== ARQUIVOS ENCONTRADOS =====")

if arquivos_clientes:
    for caminho in arquivos_clientes:
        print(caminho)
else:
    print("❌ Nenhum arquivo ecommerce_clientes encontrado na RAW")

In [0]:
from pyspark.sql.types import StructType, StructField, LongType, StringType
from pyspark.sql import functions as F

# Schema da tabela ecommerce_clientes
# Datas entram como LongType para contornar TIMESTAMP(NANOS)
schema_clientes = StructType([
    StructField("id_cliente", LongType(), True),
    StructField("uuid_cliente", StringType(), True),
    StructField("nome", StringType(), True),
    StructField("sobrenome", StringType(), True),
    StructField("email", StringType(), True),
    StructField("senha_hash", StringType(), True),
    StructField("dt_cadastro", LongType(), True),
    StructField("dt_ultima_atualizacao", LongType(), True)
])

# Origem dos dados
container_name = "raw"
tabela_alvo = "ecommerce_clientes"

file_path = f"abfss://{container_name}@{storage_account}.dfs.core.windows.net/real-time-data/*/*/*/*/{tabela_alvo}.parquet"

print("Caminho RAW:")
print(file_path)



In [0]:
# Leitura dos arquivos RAW de ecommerce_clientes

df_clientes_raw = (
    spark.read
    .options(**adls_options)
    .schema(schema_clientes)
    .parquet(file_path)
)

print("✅ RAW lido com sucesso!")
print("Total de registros:", df_clientes_raw.count())

df_clientes_raw.printSchema()
display(df_clientes_raw)

In [0]:
# Converter timestamps de nanossegundos para timestamp

df_clientes = (
    df_clientes_raw
    .withColumn(
        "dt_cadastro",
        F.timestamp_micros(
            (F.col("dt_cadastro") / 1000).cast("long")
        )
    )
    .withColumn(
        "dt_ultima_atualizacao",
        F.timestamp_micros(
            (F.col("dt_ultima_atualizacao") / 1000).cast("long")
        )
    )
)

print("✅ Datas convertidas com sucesso!")
print("Total de registros:", df_clientes.count())

df_clientes.printSchema()
display(df_clientes)

In [0]:
from pyspark.sql import functions as F

df_bronze = (
    df_clientes
    .withColumn("ano", F.year("dt_ultima_atualizacao"))
    .withColumn("mes", F.month("dt_ultima_atualizacao"))
    .withColumn("dia", F.dayofmonth("dt_ultima_atualizacao"))
)

print("✅ DataFrame preparado para gravação particionada")

display(df_bronze)

In [0]:
# Filtrar apenas registros inéditos antes de gravar na Bronze

from pyspark.sql import functions as F

try:
    # Ler IDs que já existem na Bronze
    df_ids_bronze = (
        spark.read
        .format("delta")
        .options(**adls_options)
        .load(bronze_path)
        .select("id_cliente")
        .distinct()
    )

    # Manter somente registros que ainda não existem na Bronze
    df_bronze_novos = (
        df_bronze.alias("raw")
        .join(
            df_ids_bronze.alias("bronze"),
            on="id_cliente",
            how="left_anti"
        )
    )

    print("✅ Filtro de registros inéditos concluído!")
    print("Registros recebidos:", df_bronze.count())
    print("Registros inéditos:", df_bronze_novos.count())

    display(df_bronze_novos)

except Exception as e:
    print("❌ Erro ao verificar registros inéditos:")
    print(e)

In [0]:
# Validação final da camada Bronze

df_bronze_final = (
    spark.read
    .format("delta")
    .options(**adls_options)
    .load(bronze_path)
)

total_final = df_bronze_final.count()

duplicados_final = (
    df_bronze_final
    .groupBy("id_cliente")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print("===== VALIDAÇÃO FINAL BRONZE =====")
print("Total de registros:", total_final)
print("IDs duplicados:", duplicados_final)

if total_final > 0 and duplicados_final == 0:
    print("✅ Bronze validada com sucesso!")
else:
    print("⚠️ Verificar os dados da Bronze.")

In [0]:
# ============================================================
# SALVAR CAMADA BRONZE
# ============================================================

print("Salvando dados na camada Bronze...")
print("Destino:", bronze_path)

(
    df_bronze
    .write
    .format("delta")
    .mode("overwrite")
    .options(**adls_options)
    .save(bronze_path)
)

print("✅ Camada Bronze salva com sucesso!")